# Jayhind LLM Invoice-OCR — Kaggle / Colab launcher

**Kaggle:** notebook settings → Accelerator = **GPU T4 x2**, Internet = **On**.
**Colab:** Runtime → Change runtime type → **GPU**.

Run the cells top to bottom, then keep the tab open.

**Stable URL (recommended).** Set `CF_TUNNEL_TOKEN` in cell 2 to run a **named Cloudflare tunnel** — a fixed hostname (`ocr.aakhaja.com`) that never changes across restarts. Configure the tunnel + its public hostname once in the Cloudflare Zero Trust dashboard. With a fixed `OCR_API_KEY` too, you set the hub's `.env` **once** and never touch it again.

> Leave `CF_TUNNEL_TOKEN` unset to fall back to a **quick tunnel** (new random `trycloudflare.com` URL each run).

> Either way a Cloudflare tunnel drops a single request longer than ~100s with HTTP 524 (free/pro edge cap) — a named tunnel fixes the URL, not the cap.

> **Fast repeat runs:** cell 3 caches deps + model weights to Google Drive (Colab), so a new session skips the big downloads. Skip it and everything re-downloads from zero each time.

> Cell 1 is **safe to re-run** — it resets to a clean folder first, so it can never nest.

In [ ]:
# 1. get the code (reset-safe: always clones one clean copy)
import os
base = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.chdir(base)
!rm -rf jayhind-ocr-service
!git clone https://github.com/harshbpatel11/jayhind-ocr-service.git
os.chdir(f'{base}/jayhind-ocr-service')
print('working dir:', os.getcwd())

In [ ]:
# 2. secrets — your named-tunnel token + a FIXED API key.
#    These are SECRETS. Do NOT paste real values here and commit them (this repo is public).
#    Best: add them in Colab's Secrets panel (🔑 left sidebar) as CF_TUNNEL_TOKEN and
#    OCR_API_KEY (toggle notebook access on) — this cell then reads them automatically.
import os

def _load(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, '')

os.environ['CF_TUNNEL_TOKEN'] = _load('CF_TUNNEL_TOKEN')
os.environ['OCR_API_KEY']     = _load('OCR_API_KEY')

# No Colab Secrets? Paste here for a throwaway run, then CLEAR before saving/committing:
# os.environ['CF_TUNNEL_TOKEN'] = 'eyJ...'
# os.environ['OCR_API_KEY']     = 'jayhind_ocr_...'

assert os.environ['CF_TUNNEL_TOKEN'].strip(), 'Set CF_TUNNEL_TOKEN (Colab Secrets 🔑 or paste above). Leave blank only to use a quick tunnel.'
assert os.environ['OCR_API_KEY'].strip(), 'Set OCR_API_KEY (Colab Secrets 🔑 or paste above) so the key stays fixed across restarts.'
print('secrets loaded ✓ — named tunnel, stable URL: https://ocr.aakhaja.com')

In [ ]:
# 3. persistent cache — reuse deps + model weights across sessions (fast repeat runs).
#    Colab: caches to your Google Drive (asks you to authorise the mount once).
#    Without this, every fresh session re-downloads the ~2 GB Paddle wheel + model weights.
import os

if os.path.isdir('/content'):            # Google Colab
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['CACHE_DIR'] = '/content/drive/MyDrive/jayhind-ocr-cache'
elif os.path.isdir('/kaggle'):           # Kaggle
    # /kaggle/working persists in *saved* versions (Save & Run All). For a fast
    # cold start, save this folder as a Dataset once and attach it, then point
    # CACHE_DIR at /kaggle/input/<your-dataset>.
    os.environ['CACHE_DIR'] = '/kaggle/working/jayhind-ocr-cache'

cache = os.environ.get('CACHE_DIR')
if cache:
    os.makedirs(cache, exist_ok=True)
print('CACHE_DIR =', cache or '(none — will download each session)')

In [ ]:
# 4. install + run (first run downloads a few GB; leave it running)
!bash run.sh

### Test it (open a new cell while the one above keeps running)
```python
import requests
URL = 'https://ocr.aakhaja.com'   # your stable named-tunnel URL (or the PUBLIC_URL printed above)
KEY = 'jayhind_ocr_...'           # the API_KEY printed above
print(requests.get(f'{URL}/health').json())
# r = requests.post(f'{URL}/parse',
#                   headers={'Authorization': f'Bearer {KEY}'},
#                   files={'file': open('invoice.jpg','rb')}, timeout=300)
# print(r.json())
```